# RakshakAI v2 — Training on Colab (Free T4 GPU)
This notebook trains a security-specialized 7B model using QLoRA on free T4 GPU.
Dataset: 383K samples, 631 CWEs, 30 languages


In [ ]:
# Install dependencies
!pip install -q torch==2.5.1 torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip install -q transformers==4.47.1 accelerate==1.2.1 peft==0.14.0 trl==0.15.1
!pip install -q bitsandbytes==0.45.0 datasets==3.2.0 tensorboard sentencepiece
!pip install -q axolotl==0.6.0
!pip install -q huggingface-hub

import torch
print(f'GPU: {torch.cuda.get_device_name()}' if torch.cuda.is_available() else 'CPU only')

In [ ]:
# Clone repo and download dataset
!git clone https://github.com/Muneerali199/RakshakAI.git
%cd RakshakAI

# Download dataset from HuggingFace (public)
from huggingface_hub import hf_hub_download
import os

os.makedirs('v2/inputs/datasets/axolotl', exist_ok=True)

# Download each split
repo_id = 'Muneerali199/RakshakAI-phase-b-v2'
for split in ['train.jsonl', 'val.jsonl', 'test.jsonl']:
    print(f'Downloading {split}...')
    hf_hub_download(repo_id, f'axolotl/{split}',
                    local_dir='v2/inputs/datasets', force_download=True)
    print(f'  Done')

print('Dataset ready')

In [ ]:
# Write config for T4 (QLoRA — only option on 16GB VRAM)
config = '''
base_model: Qwen/Qwen2.5-Coder-7B-Instruct

dataset:
  - path: v2/inputs/datasets/axolotl/train.jsonl
    type: jsonl
    split: train
    chat_template: chatml
    field_messages: messages

eval_dataset:
  - path: v2/inputs/datasets/axolotl/val.jsonl
    type: jsonl
    split: val
    chat_template: chatml
    field_messages: messages

val_set_size: 0
dataset_prepared_path: /content/prepared

# QLoRA for T4 (4-bit, rank 64)
load_in_4bit: true
lora_r: 64
lora_alpha: 128
lora_dropout: 0.05
lora_target_modules:
  - q_proj
  - k_proj
  - v_proj
  - o_proj
  - gate_proj
  - up_proj
  - down_proj
lora_modules_to_save:
  - embed_tokens
  - lm_head

optimizer: adamw_8bit
learning_rate: 2e-5
lr_scheduler: cosine
warmup_ratio: 0.03
num_epochs: 3

eval_strategy: steps
eval_steps: 500
eval_sample_max_num: 200

micro_batch_size: 2
gradient_accumulation_steps: 8
sequence_len: 2048
train_on_inputs: false

output_dir: /content/model
save_strategy: steps
save_steps: 1000
save_total_limit: 2

bf16: true
tf32: true
gradient_checkpointing: true
sample_packing: true
group_by_length: true

logging_steps: 10
report_to: none

special_tokens:
  pad_token: "<|endoftext|>"
seed: 42
'''

with open('colab_config.yaml', 'w') as f:
    f.write(config)
print('Config written')

In [ ]:
import subprocess
result = subprocess.run(
    ['python', '-m', 'axolotl.cli.train', 'colab_config.yaml'],
    capture_output=True, text=True
)
print(result.stdout[-2000:] if len(result.stdout) > 2000 else result.stdout)
if result.returncode != 0:
    print('STDERR:', result.stderr[-2000:])